

# **Laboratorio 10: Chatbot 101 💡**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Otoño 2026</strong></center>

### Cuerpo Docente:

- Profesores: Pablo Badilla, Diego Cortez
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### **Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados**

- Nombre de alumno 1:
- Nombre de alumno 2:

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/...../)

## **Temas a tratar**

- Large Language Models
- Output parsers
- Chatbot con RAG
- Memoria
- Análisis de embeddings

### **Objetivos principales del laboratorio**

- Resolución de problemas secuenciales usando Reinforcement Learning
- Habilitar un Chatbot para entregar respuestas útiles usando Large Language Models.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

### **0 Configuración Inicial**

<p align="center">
  <img src="https://media1.tenor.com/m/uqAs9atZH58AAAAd/config-config-issue.gif"
" width="400">
</p>

Como siempre, cargamos todas nuestras API KEY al entorno:

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

### **1. Retrieval Augmented Generation (1.0 puntos)**

#### **1.1 Reunir Documentos (0.1 puntos)**

Reuna documentos PDF sobre los que hacer preguntas siguiendo las siguientes instrucciones:
  - 2 documentos .pdf como mínimo, 5 como máximo.
  - 30 páginas de contenido como mínimo entre todos los documentos.
  - Ideas para documentos: Documentos relacionados a temas académicos, laborales o de ocio. Aprovechen este ejercicio para construir algo útil y/o relevante para ustedes!
  - Deben ocupar documentos reales, no pueden utilizar los mismos de la clase.
  - Deben registrar sus documentos en la siguiente [planilla](https://docs.google.com/spreadsheets/d/1fv7WV273_rjoFS0ORvnn-HkFYX7TCe0SNcWewwL4lkI/edit?usp=sharing). **NO PUEDEN USAR LOS MISMOS DOCUMENTOS QUE OTRO GRUPO**
  - **Recuerden adjuntar los documentos en su entrega**.

In [1]:
!uv add pyPDF2

Resolved 220 packages in 20ms
Audited 215 packages in 903ms


In [ ]:
import PyPDF2

doc_paths = []  # rellenar con los path a sus documentos

assert len(doc_paths) >= 2, "Deben adjuntar un mínimo de 2 documentos"
assert len(doc_paths) <= 5, "Deben adjuntar un máximo de 5 documentos"

total_paginas = sum(len(PyPDF2.PdfReader(open(doc, "rb")).pages) for doc in doc_paths)
assert total_paginas >= 30, f"Páginas insuficientes: {total_paginas}"

#### **1.2 Vectorizar Documentos (0.2 puntos)**

Vectorice los documentos y almacene sus representaciones de manera acorde.

#### **1.3 Habilitar RAG (0.4 puntos)**

Habilite la solución RAG a través de una **clase** que tenga un **método** `chat` que reciba la pregunta y un argumento opcional de n_results y retorne la respuesta con RAG. El resto de los argumentos debe recibirlos en la inicialización. Requisitos:
- La clase debe ser independiente, es decir no debe depender de objetos definidos fuera de ella. Todos los objetos deben recibirse como argumentos o ser generados por métodos de la clase. La  excepción son clases.
- **Requisito estricto:** el modelo generativo debe tener una temperatura de 1.0.

Luego instancie su clase y utilice el método `chat` con una pregunta de prueba. 

In [ ]:
class RAG:
    def __init__(
        self,
    ):  # Agregar argumentos a conveniencia
        ...

    def chat(self, question, n_results=5): ...

#### **1.4 Verificación de respuestas (0.2 puntos)**

Genere un listado de 3 tuplas ("pregunta", "respuesta correcta") y analice la respuesta de su solución para cada una. ¿Su solución RAG entrega las respuestas que esperaba?

Ejemplo de tupla:
- Pregunta: ¿Quién es el presidente de Chile?
- Respuesta correcta: El presidente de Chile es Gabriel Boric

#### **1.5 Persistencia de base de conocimiento (0.1 puntos)**

Guarde su base de conocimiento para reutilizarla más adelante. Su entrega deberá venir con su base de conocimiento precomputada

### **2. Creando un chatbot con RAG (2.0 puntos)**

#### **2.1 Análisis de sentimiento (0.3 puntos)**

Genere una chain que reciba una pregunta del usuario y lo clasifique según sentimiento:
- Positivo
- Neutro
- Negativo

La chain deberá retornar ESTRICTAMENTE uno de esos 3 sentimientos. Evalúe la chain con 3 ejemplos, uno para cada sentimiento.

#### **2.2 Rag con historial de chat (1.2 puntos)**

Modifique su clase (con otro nombre) que implementa RAG de forma que para generar utilice **una lista de mensajes** anteriores de la conversación. La respuesta debe considerar la conversación completa y deben haber roles claros separados en los prompts (considere la información del siguiente [enlace](https://docs.langchain.com/oss/python/langchain/messages)). Además, debe cumplir los siguientes requisitos:
- Su función de inicialización **debe** recibir los argumentos del ejemplo presente en la celda de código, con los tipos ahí presentes
- El método `chat` NO DEBE recibir la lista de mensajes. Sólo debe recibir la pregunta del usuario, n_results (opcional) y retornar la respuesta
- Debe almacenar acumulativamente tanto el **sentimiento** detectado como el **embedding** del mensaje del usuario
- Al igual que la clase que implementa RAG, debe ser independiente y el modelo debe tener una **temperatura de 1.0**
- Considere que este es un chat con memoria, por lo que deberá poder responder correctamente interacciones que no necesariamente están en el último mensaje

In [ ]:
class Chatbot:
    def __init__(
        self,
        prompts: dict,
        faiss_index_name: str,  # Pista: Qué función carga una BBDD de FAISS desde un string?
        chat_model_name: str = "gemini-3.1-flash-lite",
        embedding_model_name: str = "gemini-embedding-001",
    ): ...

    def chat(self, question, n_results=5): ...

#### **2.4 Verificación de funcionalidades de chatbot (0.5 puntos)**

Instancie e inicialice su chatbot. Luego interactúe con él con 5 a 10 mensajes donde se vean diferentes sentimientos. Cada mensaje debe llamarse en una nueva celda, donde se muestre también la respuesta del chatbot. Luego de los 10 mensajes, muestre los sentimientos detectados y el historial de chat luego de toda la interacción. 

Debe demostrar que:
- El chatbot está efectivamente usando el historial de chat para responder y no solo la última pregunta del usuario
- El chatbot está detectando efectivamente el sentimiento del usuario

In [1]:
# Inicialización chatbot

In [2]:
# Mensaje 1

In [3]:
# Mensaje 2

In [4]:
# Mensaje 3

In [5]:
# Mensaje 4

In [6]:
# Mensaje 5

In [7]:
# Mensaje 6

In [8]:
# Mensaje 7

In [9]:
# Mensaje 8

In [10]:
# Mensaje 9

In [11]:
# Mensaje 10

In [12]:
# Muestra de información recolectada

### **3. Agregando adaptabilidad al chatbot (3.0 puntos)** 

#### **3.1 Detectar intención (0.4 puntos)**

Genere las siguientes chains:
- Chain que reciba la pregunta del usuario y retorne un `booleano` que indique si la pregunta requiere o no contexto
- Chain que reciba la pregunta del usuario y retorne los valores `insolencia`, `prompt_injection` si detecta algunas de esas intenciones o `None` si no detecta ninguna.

Pruebe cada chain con ejemplos donde se obtenga cada categoría

#### **3.2 Incorporando ejemplos (0.4 puntos)**

Genere una clase ``ExampleRetriever`` que permita agregar ejemplos de pregunta / respuesta deseables a una base de conocimiento mediante la función `add_example`, y tambien obtener pares pregunta / respuesta similares a una pregunta objetivo con la función `get_examples`. Esta clase debe utilizar FAISS como base de conocimiento y **sólo utilizar la pregunta para calcular el embedding**, pero almacenar tanto la pregunta como la respuesta.

Luego, pruebe su clase con ejemplos.

Le puede ser útil investigar sobre la clase `Document` de `langchain_core.documents` y su manejo de metadatos.

In [ ]:
class ExampleRetriever:
    def __init__(self, embedding_service): ...

    def add_example(self, question, answer): ...

    def get_examples(self, question, n_examples=2): ...

#### **3.3 Mejorando el RAG (0.3 puntos)**

Si en la sección anterior solo utilizó la última pregunta del usuario para hacer retrieval, quizá puede haber notado que al preguntarle algo que referenciaba a un mensaje anterior el chatbot no siempre podía responder bien en base al conocimiento. Si utilizó los últimos mensajes del historial es menos probable que esto suceda, pero sigue sin ser lo óptimo.

Una forma de evitar este problema y también de aumentar la efectividad de RAG al disminuir la brecha semántica entre los documentos y la query es **HyDE**. Investigue sobre hyde y genere una chain o una función que genere el input necesario para ejecutar RAG con HyDE (puede serle útil [este documento](https://medium.aiplanet.com/advanced-rag-improving-retrieval-using-hypothetical-document-embeddings-hyde-1421a8ec075a))

Pruebe su función o chain con ejemplos

#### **3.4 Clase chatbot modificada (1.4 puntos)**

Genere una nueva clase de chatbot modificando su clase anterior (con otro nombre para no sobreescribirla). Su clase debe mantener las funciones principales se su chatbot como tener memoria y utilizar rag, pero debe abordar las siguientes modificaciones, utilizando las chains y funciones definidas anteriormente cuando corresponda:
- Agregar el argumento `ejemplos` al system prompt.
- Integrar ejemplos usando el método `get_examples` definido anteriormente
- Realizar HyDE para mejorar el retrieving
- Incorporar el siguiente flujo de decisiones
  - Si el sentimiento del usuario es positivo, agregar el mensaje y **respuesta anterior** a los ejemplos con `add_example`
  - Detectar si el último mensaje del usuario es insolente o intenta realizar prompt injection. Si se detecta algunas de esas intenciones, responder que no puede responder adecuadamente su pregunta y la razón de por qué. Si no, seguir con el proceso normal.
  - Sólo realizar RAG a la base de conocimientos principal si la pregunta requiere contexto
- Almacenar todas las intenciones del usuario detectadas en el flujo de decisión análogamente a como se almacenó el sentimiento del usuario. Deben guardar una relación 1<>1 entre ellas y con los mensajes del usuario
- Hacer **logging** de las decisiones tomadas en el flujo de respuesta

In [ ]:
class ChatbotAdaptable:
    def __init__(
        self,
        prompts: dict,
        faiss_index_name: str,  # Pista: Qué función carga una BBDD de FAISS desde un string?
        chat_model_name: str = "gemini-3.1-flash-lite",
        embedding_model_name: str = "gemini-embedding-001",
    ): ...

    def chat(self, question, n_results=5): ...

#### **3.5 Verificación de funcionalidades de chatbot (0.5 puntos)**

Instancie e inicialice su chatbot. Luego interactúe con él con 15 - 30 mensajes donde se vean diferentes sentimientos, intenciones y tipos de preguntas. Cada mensaje debe llamarse en una nueva celda, donde se muestre también la respuesta del chatbot. Luego de los 15 mensajes, muestre los sentimientos detectados. 

Debe demostrar que:
- El chatbot está efectivamente usando el historial de chat para responder
- El chatbot es capaz de responder con conocimiento incluso cuando el último mensaje no menciona su pregunta explícitamente (ej. 'Cuéntame más')
- El chatbot no responde cuando se le habla insolentemente o se intenta realizar prompt_injection
- El chatbot no realiza rag si la pregunta puede responderse directamente (o no es una pregunta)
- El chatbot agrega ejemplos cuando el usuario hace una pregunta con sentimiento positivo.

In [13]:
# Inicialización chatbot

In [14]:
# Mensaje 1

In [15]:
# Mensaje 2

In [16]:
# Mensaje 3

In [17]:
# Mensaje 4

In [18]:
# Mensaje 5

In [19]:
# Mensaje 6

In [20]:
# Mensaje 7

In [21]:
# Mensaje 8

In [22]:
# Mensaje 9

In [23]:
# Mensaje 10

In [24]:
# Mensaje 11

In [25]:
# Mensaje 12

In [26]:
# Mensaje 13

In [27]:
# Mensaje 14

In [28]:
# Mensaje 15

### **4. Análisis semántico (Bonus + 0.5 puntos)**

Visualice la distribución semántica de los mensajes almacenados en el chatbot utilizando alguna técnica de reducción de dimensionalidad sobre los embeddings. Haga 2 gráficos de dispersión: uno coloreado por sentimiento y otro coloreado por si la pregunta requiere o no contexto. Luego responda:

¿Observa algun patrón de agrupación? ¿A qué puede deberse?

**Tip:** Para esta actividad puede generar más llamadas al chatbot (no es necesario mostrar las respuestas) y así completar categorías que pueden estar menos representadas y aumentar la cantidad de datos, con lo que funciona mejor el modelo de reducción de dimensionalidad

In [ ]:
# Código procesamiento y gráficos

**Respuesta:**